# Insights e Recomendacoes — Supply Chain Ball Corporation

Narrativa: **qualidade de dados → desalinhamento SLA/lead time → desequilibrio geografico de estoque → causalidade stockout/atraso → recomendacoes**

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT   = Path('..')
OBT    = ROOT / 'data' / 'gold' / 'obt.parquet'
SILVER = ROOT / 'data' / 'silver'

conn = duckdb.connect()
conn.execute(f"CREATE VIEW obt AS SELECT * FROM read_parquet('{OBT}')")

sns.set_theme(style='whitegrid', palette='muted')

## Contexto: Qualidade dos Dados

Tres problemas de qualidade afetam a confiabilidade dos indicadores:

| Problema | Impacto |
|---|---|
| Pedidos sem `actual_delivery_date` | OTIF subestimado — tratados como nao entregues no prazo |
| `order_id` duplicados | Contagem inflada antes da deduplicacao |
| `stock_level` negativo | Backorder nao capturado como stockout em sistemas legados |

In [ ]:
qualidade = conn.execute("""
    SELECT
        SUM(total_pedidos)                                      AS total_pedidos,
        SUM(pedidos_sem_entrega)                                AS sem_data_entrega,
        ROUND(100.0 * SUM(pedidos_sem_entrega)
              / NULLIF(SUM(total_pedidos), 0), 1)               AS pct_sem_entrega,
        SUM(CASE WHEN stock_level < 0 THEN 1 ELSE 0 END)       AS dias_estoque_negativo
    FROM obt
""").df()

print(qualidade.to_string(index=False))

## Insight 1 — O SLA nao reflete a capacidade operacional real

O OTIF e sistematicamente baixo em todas as regioes e periodos. A causa principal nao e stockout nem falha pontual: e o desalinhamento entre o SLA contratado e o lead time real da operacao.

In [ ]:
# OTIF geral e por regiao
otif_geral = conn.execute("""
    SELECT
        ROUND(100.0 * SUM(pedidos_on_time) / NULLIF(SUM(total_pedidos), 0), 2) AS otif_pct,
        ROUND(AVG(CASE WHEN total_pedidos > 0 THEN atraso_medio END), 2)        AS atraso_medio_dias
    FROM obt WHERE total_pedidos > 0
""").df()

otif_regiao = conn.execute("""
    SELECT region,
        ROUND(100.0 * SUM(pedidos_on_time) / NULLIF(SUM(total_pedidos), 0), 2) AS otif_pct
    FROM obt WHERE total_pedidos > 0
    GROUP BY region ORDER BY otif_pct
""").df()

otif_trimestral = conn.execute("""
    SELECT
        CONCAT('Q', QUARTER(date)) AS trimestre,
        ROUND(100.0 * SUM(pedidos_on_time) / NULLIF(SUM(total_pedidos), 0), 2) AS otif_pct
    FROM obt WHERE total_pedidos > 0
    GROUP BY trimestre ORDER BY trimestre
""").df()

media = otif_geral['otif_pct'].iloc[0]
print(f"OTIF geral: {media}%")
print(f"Amplitude entre regioes: {otif_regiao['otif_pct'].min()}% - {otif_regiao['otif_pct'].max()}%")
print(f"Atraso medio: {otif_geral['atraso_medio_dias'].iloc[0]:.1f} dias alem do SLA")

fig, eixos = plt.subplots(1, 2, figsize=(12, 4))

cores = ['tomato' if v < media else 'steelblue' for v in otif_regiao['otif_pct']]
eixos[0].barh(otif_regiao['region'], otif_regiao['otif_pct'], color=cores)
eixos[0].axvline(media, color='black', linestyle='--', linewidth=1)
eixos[0].set_title('OTIF por regiao (vermelho = abaixo da media)')
eixos[0].set_xlabel('OTIF (%)')

cores_t = ['tomato' if v < media else 'steelblue' for v in otif_trimestral['otif_pct']]
eixos[1].bar(otif_trimestral['trimestre'], otif_trimestral['otif_pct'], color=cores_t)
eixos[1].axhline(media, color='black', linestyle='--', linewidth=1)
eixos[1].set_title('OTIF por trimestre (media agregada)')
eixos[1].set_ylabel('OTIF (%)')

plt.tight_layout()
plt.show()

In [ ]:
# Confirmacao do SLA e lead time real
sla_check = conn.execute(f"""
    SELECT
        MIN(DATEDIFF('day', order_date, requested_delivery_date)) AS sla_min,
        MAX(DATEDIFF('day', order_date, requested_delivery_date)) AS sla_max,
        ROUND(AVG(DATEDIFF('day', order_date, actual_delivery_date)), 1) AS lead_time_real_medio,
        ROUND(AVG(DATEDIFF('day', order_date, requested_delivery_date)), 1) AS sla_medio
    FROM read_parquet('{SILVER}/orders.parquet')
    WHERE actual_delivery_date IS NOT NULL
""").df()

print("SLA confirmado nos dados:")
print(sla_check.to_string(index=False))
print()
print(f"Diferenca (lead time real - SLA): {sla_check['lead_time_real_medio'].iloc[0] - sla_check['sla_medio'].iloc[0]:.1f} dias em media")

In [ ]:
# Simulacao: qual seria o OTIF para diferentes valores de SLA?
sla_vals = [5, 6, 7, 8, 10]
resultados = []
total_ped = conn.execute("SELECT SUM(total_pedidos) FROM obt WHERE total_pedidos > 0").fetchone()[0]
on_time   = conn.execute("SELECT SUM(pedidos_on_time) FROM obt WHERE total_pedidos > 0").fetchone()[0]

for sla in sla_vals:
    extra = conn.execute(f"""
        SELECT SUM(pedidos_com_atraso)
        FROM obt
        WHERE total_pedidos > 0 AND atraso_medio <= {sla - 5}
    """).fetchone()[0] or 0
    otif_sim = round(100 * (on_time + extra) / total_ped, 1)
    resultados.append({'sla': f"{sla}d", 'otif': otif_sim})

df_sla = pd.DataFrame(resultados)

cores_sla = ['tomato' if s == '5d' else 'steelblue' for s in df_sla['sla']]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(df_sla['sla'], df_sla['otif'], color=cores_sla)
for i, row in df_sla.iterrows():
    ax.text(i, row['otif'] + 0.5, f"{row['otif']}%", ha='center', fontsize=9)
ax.set_title('OTIF simulado por valor de SLA')
ax.set_xlabel('SLA (dias)')
ax.set_ylabel('OTIF (%)')
ax.set_ylim(0, max(df_sla['otif']) * 1.15)
plt.tight_layout()
plt.show()

print(df_sla.to_string(index=False))

**Analise:** A amplitude do OTIF entre regioes e pequena — o problema nao e localizado. O SLA contratado e confirmado nos dados (todos os pedidos tem `requested_delivery_date = order_date + SLA`). O lead time real medio e superior ao SLA, gerando um atraso medio sistematico alem do prazo prometido.

**Causa provavel:** O SLA foi definido sem considerar o lead time real da operacao com producao regionalizada e estoques descentralizados.

**Recomendacao:** Revisar o SLA com base na distribuicao real de lead times. A simulacao mostra o impacto quantitativo de diferentes cenarios. Esta e possivelmente a acao de maior impacto imediato no OTIF, sem exigir mudancas operacionais.

## Insight 2 — O estoque esta no lugar errado

Ha regioes com stockout cronico e outras com overflow para o mesmo produto no mesmo periodo. O problema nao e falta de producao — e falta de redistribuicao.

In [ ]:
# Stockout e overflow por regiao
so_reg = conn.execute("""
    SELECT region,
        ROUND(100.0 * SUM(CASE WHEN stockout_flag  THEN 1 ELSE 0 END) / COUNT(*), 1) AS stockout_pct,
        ROUND(100.0 * SUM(CASE WHEN overflow_flag  THEN 1 ELSE 0 END) / COUNT(*), 1) AS overflow_pct,
        ROUND(AVG(stock_level), 0)                                                    AS stock_medio
    FROM obt
    GROUP BY region ORDER BY stockout_pct DESC
""").df()

print(so_reg.to_string(index=False))

regioes = so_reg['region']
x = range(len(regioes))
largura = 0.35
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar([i - largura/2 for i in x], so_reg['stockout_pct'], largura, label='Stockout (%)', color='tomato')
ax.bar([i + largura/2 for i in x], so_reg['overflow_pct'], largura, label='Overflow (%)', color='orange')
ax.set_xticks(list(x))
ax.set_xticklabels(regioes)
ax.set_title('Stockout vs Overflow por regiao (% dos dias)')
ax.set_ylabel('%')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Desequilibrio simultaneo: mesma data, mesmo produto, regiao A overflow e regiao B stockout
deseq = conn.execute("""
    WITH pares AS (
        SELECT
            a.date, a.product,
            a.region AS regiao_overflow, b.region AS regiao_stockout,
            a.stock_level AS estoque_excesso
        FROM obt a
        JOIN obt b
            ON a.date = b.date
            AND a.product = b.product
            AND a.region <> b.region
            AND a.overflow_flag = TRUE
            AND b.stockout_flag = TRUE
    )
    SELECT
        product,
        regiao_overflow || ' -> ' || regiao_stockout AS par,
        COUNT(*)                               AS dias_simultaneos,
        ROUND(AVG(estoque_excesso), 0)         AS estoque_medio_parado
    FROM pares
    GROUP BY product, par
    ORDER BY dias_simultaneos DESC
""").df()

print("Desequilibrio simultaneo por par de regioes:")
print(deseq.to_string(index=False))

**Analise:** Stockout e overflow co-ocorrem para o mesmo produto em regioes distintas no mesmo dia. O modelo de producao regionalizada (cada planta abastece sua propria regiao) nao e adequado para a distribuicao real da demanda.

**Causa provavel:** Ausencia de mecanismo de transferencia entre regioes. O excesso em uma regiao nao compensa a falta em outra.

**Recomendacao:** Avaliar transferencias inter-regionais para os pares com maior volume de desequilibrio. Possivelmente executavel sem aumento de producao, sujeito a validacao operacional.

## Insight 3 — Stockout co-ocorre com atraso: indicio de causalidade

O join por `order_date` captura apenas se havia stockout no dia exato do pedido. O range join (pedido x inventario na janela `order_date → actual_delivery_date`) e o metodo correto para avaliar o impacto real do stockout no ciclo de entrega.

In [ ]:
# Co-ocorrencia temporal: stockout% e atraso medio por trimestre
causal = conn.execute("""
    SELECT
        CONCAT('Q', QUARTER(date)) AS trimestre,
        ROUND(100.0 * AVG(CASE WHEN stockout_flag THEN 1.0 ELSE 0.0 END), 1) AS stockout_pct,
        ROUND(AVG(atraso_medio), 2)                                            AS atraso_medio
    FROM obt
    GROUP BY trimestre ORDER BY trimestre
""").df()

fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()
ax1.bar(causal['trimestre'], causal['stockout_pct'], color='tomato', alpha=0.7, label='Stockout%')
ax2.plot(causal['trimestre'], causal['atraso_medio'], color='steelblue', marker='o', linewidth=2, label='Atraso medio (dias)')
ax1.set_ylabel('Stockout (%)')
ax2.set_ylabel('Atraso medio (dias)')
ax1.set_title('Stockout e atraso co-ocorrem por trimestre')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Range join: para cada pedido, verifica se houve stockout em qualquer dia do ciclo order->delivery
impacto = conn.execute(f"""
    WITH ordens AS (
        SELECT order_id, region, product, order_date, actual_delivery_date, on_time
        FROM read_parquet('{SILVER}/orders.parquet')
        WHERE actual_delivery_date IS NOT NULL
    ),
    dias_stockout AS (
        SELECT date, region, product
        FROM read_parquet('{SILVER}/inventory.parquet')
        WHERE stockout_flag = TRUE
    ),
    pedidos_com_stockout AS (
        SELECT DISTINCT o.order_id
        FROM ordens o
        INNER JOIN dias_stockout s
            ON o.region = s.region
            AND o.product = s.product
            AND s.date >= o.order_date
            AND s.date <= o.actual_delivery_date
    )
    SELECT
        CASE WHEN pcs.order_id IS NOT NULL THEN 'com_stockout_no_ciclo' ELSE 'sem_stockout_no_ciclo' END AS grupo,
        COUNT(*)                                                                                          AS pedidos,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)                                               AS pct_total,
        ROUND(100.0 * AVG(CASE WHEN o.on_time THEN 1.0 ELSE 0.0 END), 1)                                AS otif
    FROM ordens o
    LEFT JOIN pedidos_com_stockout pcs ON o.order_id = pcs.order_id
    GROUP BY 1 ORDER BY 1
""").df()

print(impacto.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 3))
cores_g = ['tomato', 'steelblue']
ax.barh(impacto['grupo'], impacto['otif'], color=cores_g)
for i, row in impacto.iterrows():
    ax.text(row['otif'] + 0.3, i, f"{row['otif']}%  ({row['pedidos']} pedidos, {row['pct_total']}%)", va='center', fontsize=9)
ax.set_title('OTIF por grupo: com vs sem stockout no ciclo de entrega')
ax.set_xlabel('OTIF (%)')
ax.set_xlim(0, 35)
plt.tight_layout()
plt.show()

In [ ]:
# Simulacao do impacto global no OTIF se rupturas fossem eliminadas
# Base: OBT (inclui pedidos sem data de entrega como falha, consistente com o OTIF reportado)
on_time_total = conn.execute("SELECT SUM(pedidos_on_time) FROM obt WHERE total_pedidos > 0").fetchone()[0]
total_pedidos = conn.execute("SELECT SUM(total_pedidos)  FROM obt WHERE total_pedidos > 0").fetchone()[0]

otif_so  = float(impacto.loc[impacto['grupo'] == 'com_stockout_no_ciclo',  'otif'].iloc[0]) / 100
otif_nso = float(impacto.loc[impacto['grupo'] == 'sem_stockout_no_ciclo', 'otif'].iloc[0]) / 100
n_so     = int(impacto.loc[impacto['grupo']   == 'com_stockout_no_ciclo',  'pedidos'].iloc[0])

extra_on_time  = n_so * (otif_nso - otif_so)
otif_atual     = round(100 * on_time_total / total_pedidos, 1)
otif_potencial = round(100 * (on_time_total + extra_on_time) / total_pedidos, 1)

print(f"OTIF atual (global):     {otif_atual}%")
print(f"OTIF potencial (global): {otif_potencial}%")
print(f"Ganho estimado:          +{round(otif_potencial - otif_atual, 1)}pp")
print()
print("Nota: assume que a associacao stockout/atraso e causal e que eliminar o stockout")
print("elevaria o grupo afetado para a taxa do grupo sem stockout. Sujeito a validacao operacional.")

**Analise:** O join por `order_date` (abordagem original) capturava apenas stockouts no dia exato do pedido, subestimando o impacto. O range join identifica pedidos que passaram por pelo menos 1 dia de stockout durante o ciclo completo de entrega — um grupo com OTIF significativamente inferior ao dos pedidos sem stockout.

O grafico de co-ocorrencia temporal sugere que stockout e atraso caminham juntos nos mesmos periodos, o que reforca o indicio de causalidade. Os dados nao provam causalidade direta, mas o padrao justifica investigar stockout como possivel causa raiz dos atrasos graves.

**Recomendacao:** Resolver o desequilibrio geografico (Insight 2) possivelmente reduziria o stockout e, se a causalidade se confirmar, elevaria o OTIF global.

## Insight 4 — Opacidade operacional: pedidos sem data de entrega

Uma parcela dos pedidos nao tem `actual_delivery_date` registrado. Sem campo de status do pedido, nao e possivel distinguir pedidos entregues (sem registro) de pedidos em transito ou cancelados.

In [ ]:
sem_entrega = conn.execute("""
    SELECT region, product, SUM(pedidos_sem_entrega) AS sem_entrega
    FROM obt
    GROUP BY region, product
    HAVING SUM(pedidos_sem_entrega) > 0
    ORDER BY sem_entrega DESC
""").df()

total_sem = int(sem_entrega['sem_entrega'].sum())
total_ped = int(conn.execute('SELECT SUM(total_pedidos) FROM obt').fetchone()[0])
print(f"Pedidos sem data de entrega: {total_sem} ({100 * total_sem / total_ped:.1f}% do total)")
print(f"Distribuicao uniforme entre regioes e produtos (sem concentracao especifica):")
print(sem_entrega.to_string(index=False))

**Analise:** Os pedidos sem data de entrega estao distribuidos entre todas as regioes e produtos, sem concentracao que sugira falha de integracao especifica. O OTIF reportado possivelmente subestima o problema real.

**Recomendacao:** Adicionar campo `status_pedido` ao sistema de ordens (entregue, em transito, cancelado). Pre-requisito para calcular OTIF com precisao.

## Priorizacao das Recomendacoes

| Prioridade | Recomendacao | Racional |
|---|---|---|
| Alta | Revisar SLA | Principal driver do OTIF baixo. Realinhar o contrato com a capacidade operacional real e possivelmente a acao de maior impacto imediato. |
| Alta | Implementar `status_pedido` | Elimina opacidade nos pedidos sem data de entrega. O OTIF real pode ser diferente do medido. |
| Media | Redistribuir estoque entre regioes | Stockout e overflow co-ocorrem no mesmo produto. Possivelmente executavel sem aumento de producao. Se a causalidade se confirmar, impacto estimado na faixa de +2 a +3pp no OTIF global. |
| Media | Politica de reabastecimento diferenciada | Cada produto e regiao tem perfil de stockout distinto. Politica unica possivelmente subaproveita a previsibilidade dos dados historicos. |
| Baixa | Visibilidade compartilhada de estoque | Indicio de operacao descoordenada entre regioes. Permitiria redistribuicao proativa antes da ruptura. |
| Baixa | Auditar registros de estoque negativo e capacidade | Inconsistencias possivelmente distorcem analises de overflow e projecao de capacidade. |

In [ ]:
conn.close()